# Save Final Gradient Boosting Model

This notebook trains the selected final model for predicting `migraine_occurrence` and saves both the trained model and the feature list for later use in the project.

## 1. Import required libraries

The following libraries are used for data handling, model training, model evaluation, and saving the trained model files.

In [1]:
from pathlib import Path
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 2. Load the dataset

The dataset is loaded using the full file path so that the notebook runs reliably in this VS Code setup.

In [2]:
data_path = Path('/Users/cioran/Documents/migraine-prediction-project/data/migraine_dataset.csv')
print('Loading dataset from:', data_path)
df = pd.read_csv(data_path)
df.head()

Loading dataset from: /Users/cioran/Documents/migraine-prediction-project/data/migraine_dataset.csv


,user_id,date,sleep_hours,mood_level,stress_level,hydration_level,screen_time,migraine_occurrence,migraine_severity
0,1,1/15/2024,7.8,3,2,2,4.7,1,1
1,1,1/16/2024,6.6,4,1,2,3.2,1,1
2,1,1/17/2024,8.5,4,2,2,4.7,1,2
3,1,1/18/2024,7.5,3,2,3,3.8,1,3
4,1,1/19/2024,9.0,3,2,1,6.8,1,2


## 3. Convert the `date` column to datetime

The `date` column is converted to datetime format so that date-based features can be created.

In [3]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dtype)

datetime64[us]


## 4. Create the improved features

This step creates the date-based features and interaction features that were used in the model improvement stage.

In [4]:
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['sleep_stress_interaction'] = df['sleep_hours'] * df['stress_level']
df['screen_stress_interaction'] = df['screen_time'] * df['stress_level']
df['hydration_stress_interaction'] = df['hydration_level'] * df['stress_level']

df[[
    'day_of_week',
    'month',
    'sleep_stress_interaction',
    'screen_stress_interaction',
    'hydration_stress_interaction'
]].head()

,day_of_week,month,sleep_stress_interaction,screen_stress_interaction,hydration_stress_interaction
0,0,1,15.6,9.4,4
1,1,1,6.6,3.2,2
2,2,1,17.0,9.4,4
3,3,1,15.0,7.6,6
4,4,1,18.0,13.6,2


## 5. Define `X` and `y`

The feature matrix `X` contains the improved feature set, and `y` contains the target variable `migraine_occurrence`.

In [5]:
feature_names = [
    'sleep_hours',
    'mood_level',
    'stress_level',
    'hydration_level',
    'screen_time',
    'day_of_week',
    'month',
    'sleep_stress_interaction',
    'screen_stress_interaction',
    'hydration_stress_interaction'
]

target = 'migraine_occurrence'

X = df[feature_names]
y = df[target]

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

Shape of X: (11879, 10)
Shape of y: (11879,)


## 6. Split the data into training and testing sets

The dataset is split into training and testing subsets so that the model can be evaluated before being retrained on the full dataset.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

X_train shape: (9503, 10)
X_test shape: (2376, 10)
y_train shape: (9503,)
y_test shape: (2376,)


## 7. Train a Gradient Boosting model

Gradient Boosting is trained because it was the strongest overall model during the model improvement stage.

In [7]:
gb_model = GradientBoostingClassifier(random_state=42)
gb_model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

## 8. Evaluate the model

The model is evaluated on the test set using standard classification metrics.

In [8]:
y_pred = gb_model.predict(X_test)
y_prob = gb_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print('Gradient Boosting Performance')
print('Accuracy:', accuracy)
print('Precision:', precision)
print('Recall:', recall)
print('F1-score:', f1)
print('ROC-AUC:', roc_auc)
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Gradient Boosting Performance
Accuracy: 0.6102693602693603
Precision: 0.6038602941176471
Recall: 0.5703125
F1-score: 0.5866071428571429
ROC-AUC: 0.6412448370551924

Confusion Matrix:
[[793 431]
 [495 657]]

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.65      0.63      1224
           1       0.60      0.57      0.59      1152

    accuracy                           0.61      2376
   macro avg       0.61      0.61      0.61      2376
weighted avg       0.61      0.61      0.61      2376



## 9. Train the final model on the full dataset

After evaluation, the final Gradient Boosting model is trained again using the full dataset so that all available information is used before saving the model.

In [9]:
final_model = GradientBoostingClassifier(random_state=42)
final_model.fit(X, y)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

## 10. Save the final model

The trained final model is saved in the `models` folder using `joblib`.

In [10]:
model_path = Path('/Users/cioran/Documents/migraine-prediction-project/models/migraine_gradient_boosting_model.joblib')
joblib.dump(final_model, model_path)
print('Saved model to:', model_path)

Saved model to: /Users/cioran/Documents/migraine-prediction-project/models/migraine_gradient_boosting_model.joblib


## 11. Save the feature list

The feature names are saved separately so that the backend can later reconstruct the expected input order for predictions.

In [11]:
feature_path = Path('/Users/cioran/Documents/migraine-prediction-project/models/feature_names.joblib')
joblib.dump(feature_names, feature_path)
print('Saved feature names to:', feature_path)

Saved feature names to: /Users/cioran/Documents/migraine-prediction-project/models/feature_names.joblib


## 12. Final note

The saved model and feature list will later be loaded by the backend API so that the web application can generate migraine occurrence predictions from user input.